# DATA CLEANING
# 1.Loading Datasets
### Load Raw Spotify Data
- Import the three raw Spotify export files: a saved playlist (`Playlist1.json`),the streaming history (`StreamingHistory_music_0.json`), and the user's savedlibrary (`YourLibrary.json`). 
- The playlist and streaming history are loaded directly into DataFrames, while the library file is loaded as raw JSON sinceits structure needs to be unpacked manually later.

In [ ]:
# Loading Dataset
import pandas as pd
import json

df1 = pd.read_json(r"Data/Raw/Playlist1.json")

df2 = pd.read_json(r"Data/Raw/StreamingHistory_music_0.json")

library_path = r"Data/Raw/YourLibrary.json"

with open(library_path, "r", encoding="utf-8") as f:
    library_data = json.load(f)

## 2. Quick Sanity Check
Print the shape of each raw dataset to confirm the files loaded correctly and to get a first sense of how much data is available before cleaning.

In [19]:
print("Shape of Playlist DataFrame:", df1.shape)
print("Shape of StreamingHistory DataFrame:", df2.shape)
print("Number of items in YourLibrary JSON:", len(library_data))


Shape of Playlist DataFrame: (16, 1)
Shape of StreamingHistory DataFrame: (8229, 4)
Number of items in YourLibrary JSON: 13


## 3. Clean the Streaming History Table
Make a working copy of the streaming data and rename Spotify's camelCase columns (`endTime`, `artistName`, etc.) to consistent snake_case names that will be used throughout the rest of the project.

In [20]:
# Streaming History DataFrame Cleaning
streaming_df = df2.copy()

streaming_df = streaming_df.rename(columns={
    "endTime": "end_time",
    "artistName": "artist_name",
    "trackName": "track_name",
    "msPlayed": "ms_played"
})

streaming_df.head()

,end_time,artist_name,track_name,ms_played
0,2025-06-20 20:20,Charlie Puth,Attention,208786
1,2025-06-20 20:25,Ed Sheeran,Photograph,258987
2,2025-06-20 20:28,Ariana Grande,One Last Time,197266
3,2025-06-20 20:32,Camila Cabello,Never Be the Same,226974
4,2025-06-20 20:36,James Arthur,Car's Outside,248373


## 4. Engineer Time Features and Build the Track Key
- Convert `end_time` to a proper datetime, calculate minutes played from milliseconds and derive date/year/month/hour/weekday columns for later time-based analysis. 
- Strip whitespace from artist and track names andbuild a standardized `track_key` (`"artist - track"`, lowercased) that will be used to match the same song across the streaming, playlist and library tables. 
- Finally, drop rows with missing core fields and remove exact duplicate plays.

In [21]:
# Convert date and listening time
streaming_df["end_time"] = pd.to_datetime(streaming_df["end_time"], errors="coerce")
streaming_df["minutes_played"] = streaming_df["ms_played"] / 60000

# Create time columns
streaming_df["date"] = streaming_df["end_time"].dt.date
streaming_df["year"] = streaming_df["end_time"].dt.year
streaming_df["month"] = streaming_df["end_time"].dt.month
streaming_df["hour"] = streaming_df["end_time"].dt.hour
streaming_df["weekday"] = streaming_df["end_time"].dt.day_name()

# Clean text
streaming_df["artist_name"] = streaming_df["artist_name"].str.strip()
streaming_df["track_name"] = streaming_df["track_name"].str.strip()

# Creating Track key
streaming_df["track_key"] = (
    streaming_df["artist_name"].str.lower()
    + " - "
    + streaming_df["track_name"].str.lower()
)

# Removing duplicates and null values
print("Null values before cleaning:\n", streaming_df.isnull().sum())
print("Duplicates before cleaning:", streaming_df.duplicated().sum())
streaming_df = streaming_df.dropna(subset=["artist_name", "track_name", "end_time"])
streaming_df = streaming_df.drop_duplicates()

print(streaming_df.tail())

Null values before cleaning:
 end_time          0
artist_name       0
track_name        0
ms_played         0
minutes_played    0
date              0
year              0
month             0
hour              0
weekday           0
track_key         0
dtype: int64
Duplicates before cleaning: 0
                end_time artist_name            track_name  ms_played  \
8224 2026-06-20 10:13:00  Chris Grey  US AGAINST THE WORLD      21750   
8225 2026-06-20 10:16:00       Naïka                  6:45     202134   
8226 2026-06-20 10:17:00      Miguel            Sure Thing      20884   
8227 2026-06-20 10:27:00      Masego                 Tadow       3925   
8228 2026-06-20 20:02:00  Max Allais   When The Party Ends     141184   

      minutes_played        date  year  month  hour   weekday  \
8224        0.362500  2026-06-20  2026      6    10  Saturday   
8225        3.368900  2026-06-20  2026      6    10  Saturday   
8226        0.348067  2026-06-20  2026      6    10  Saturday   
8227    

## 5. Playlist DataFrame Cleaning
### Flatten the Playlist JSON into a Table
The raw playlist data is nested (a playlist contains a list of track items), so loop through each playlist and each track inside it to build a flat list of rows — one row per (playlist, track) pair — capturing the playlist name,last-modified date, when the track was added and basic track/album info.

In [22]:
# Playlist DataFrame Cleaning
# Playlist table
playlist_rows = []

for playlist in df1["playlists"]:
    playlist_name = playlist.get("name")
    playlist_last_modified = playlist.get("lastModifiedDate")

    for item in playlist.get("items", []):
        track = item.get("track", {})

        playlist_rows.append({
            "playlist_name": playlist_name,
            "playlist_last_modified": playlist_last_modified,
            "added_date": item.get("addedDate"),
            "track_name": track.get("trackName"),
            "artist_name": track.get("artistName"),
            "album_name": track.get("albumName"),
            "track_uri": track.get("trackUri")
        })

playlist_df = pd.DataFrame(playlist_rows)

playlist_df.head()

,playlist_name,playlist_last_modified,added_date,track_name,artist_name,album_name,track_uri
0,Sensual,2026-06-20,2026-06-20,SHOOK,H33RA,SHOOK,spotify:track:0wdSmyHGxoNaJmLENqcaUj
1,Sensual,2026-06-20,2026-06-20,Best Mistake,Ariana Grande,My Everything,spotify:track:3uDFCbWt1T19sz8zhBuaUc
2,Sensual,2026-06-20,2026-06-20,Sex Talk,D4M $loan,I Couldn't Decide,spotify:track:0X2eq8W4o9qLoOAMil6Hce
3,Sensual,2026-06-20,2026-06-20,Avenue,H.E.R.,H.E.R. Volume 2,spotify:track:3qSH4NSYYOMkFTGwvGA91x
4,Sensual,2026-06-20,2026-06-20,safety net (feat. Ty Dolla $ign),Ariana Grande,Positions,spotify:track:14gkWVFMwdxBMyqBw1wmIg


## 6. Convert date columns
### Clean the Playlist Table
Convert the playlist's date columns to proper datetimes, strip whitespace from all text columns and build the same `track_key` format used in the streaming table so playlist tracks can later be joined to streaming and library data. Drop rows missing artist/track info and remove duplicates.

In [23]:
# Convert date columns
playlist_df["playlist_last_modified"] = pd.to_datetime(
    playlist_df["playlist_last_modified"],
    errors="coerce"
)

playlist_df["added_date"] = pd.to_datetime(
    playlist_df["added_date"],
    errors="coerce"
)

# Clean text columns
playlist_text_columns = [
    "playlist_name",
    "track_name",
    "artist_name",
    "album_name"
]

for col in playlist_text_columns:
    playlist_df[col] = playlist_df[col].astype(str).str.strip()

# Creating Track key
playlist_df["track_key"] = (
    playlist_df["artist_name"].str.lower()
    + " - "
    + playlist_df["track_name"].str.lower()
)

# Removing duplicates and null values
print("Null values before cleaning:\n", playlist_df.isnull().sum())
print("Duplicates before cleaning:", playlist_df.duplicated().sum())
playlist_df = playlist_df.dropna(subset=["artist_name", "track_name"])
playlist_df = playlist_df.drop_duplicates()

print(playlist_df.tail())

Null values before cleaning:
 playlist_name             0
playlist_last_modified    0
added_date                0
track_name                0
artist_name               0
album_name                0
track_uri                 0
track_key                 0
dtype: int64
Duplicates before cleaning: 0
     playlist_name playlist_last_modified added_date  \
1092         KIARA             2026-01-28 2025-03-03   
1093         KIARA             2026-01-28 2025-03-03   
1094         KIARA             2026-01-28 2025-03-03   
1095         KIARA             2026-01-28 2025-06-21   
1096         KIARA             2026-01-28 2026-01-28   

                                          track_name     artist_name  \
1092                                  Constellations      Jade LeMac   
1093                                        Firework      Katy Perry   
1094                                        I.F.L.Y.           Bazzi   
1095     Like I'm Gonna Lose You (feat. John Legend)  Meghan Trainor   
1096  

## 7. Library Data Cleaning
### Clean the Saved Library Table
Convert the library JSON's `tracks` list into a DataFrame, rename its columns to match the naming convention used elsewhere, clean the text fields and build the matching `track_key`. Add an `in_library` flag (always True here, since every row in this table is a saved song) and drop missing/duplicate rows.


In [24]:
# Library Data Cleaning
library_df = pd.DataFrame(library_data["tracks"])

library_df.head()

# Rename columns for consistency
library_df = library_df.rename(columns={
    "artist": "artist_name",
    "album": "album_name",
    "track": "track_name",
    "uri": "track_uri"
})

# Clean text columns
library_text_columns = [
    "artist_name",
    "album_name",
    "track_name"
]

for col in library_text_columns:
    library_df[col] = library_df[col].astype(str).str.strip()

 # Creating Track key
library_df["track_key"] = (
    library_df["artist_name"].str.lower()
    + " - "
    + library_df["track_name"].str.lower()
)   

# Mark saved songs
library_df["in_library"] = True

# Removing duplicates and null values
print("Null values before cleaning:\n", library_df.isnull().sum())
print("Duplicates before cleaning:", library_df.duplicated().sum())
library_df = library_df.dropna(subset=["artist_name", "track_name"])
library_df = library_df.drop_duplicates()

print(library_df.tail())


Null values before cleaning:
 artist_name    0
album_name     0
track_name     0
track_uri      0
track_key      0
in_library     0
dtype: int64
Duplicates before cleaning: 0
       artist_name     album_name   track_name  \
297  Ariana Grande  thank u, next        needy   
298   James Arthur            YOU  Empty Space   
299           SWIM      Body Loud    Body Loud   
300        ENHYPEN   ORANGE BLOOD  Sweet Venom   
301     Tory Lanez     I Told You       Say It   

                                track_uri                   track_key  \
297  spotify:track:1TEL6MlSSVLSdhOSddidlJ       ariana grande - needy   
298  spotify:track:6GFfDRG1Sff3IZaD0TYro4  james arthur - empty space   
299  spotify:track:1dKyceHSpbmHEghVvhTrfo            swim - body loud   
300  spotify:track:2YmfV4lAjrAQvuggKCUX6m       enhypen - sweet venom   
301  spotify:track:2Gyc6e2cLxA5hoX1NOvYnU         tory lanez - say it   

     in_library  
297        True  
298        True  
299        True  
300        Tr

## 8. Check Table Shape and preview them
### Preview All Three Cleaned Tables
Print the shape and a preview of the streaming, playlist and library tables side by side to confirm the cleaning steps worked as expected before moving on to summarization.

In [25]:
# Check Table Shape and preview them 
print("Streaming table:", streaming_df.shape)
print("Playlist table:", playlist_df.shape)
print("Library table:", library_df.shape)

print( "Streaming Table Preview:\n",streaming_df.head())
print( "Playlist Table Preview:\n",playlist_df.head())
print( "Library Table Preview:\n",library_df.head())


Streaming table: (8229, 11)
Playlist table: (1097, 8)
Library table: (302, 6)
Streaming Table Preview:
              end_time     artist_name         track_name  ms_played  \
0 2025-06-20 20:20:00    Charlie Puth          Attention     208786   
1 2025-06-20 20:25:00      Ed Sheeran         Photograph     258987   
2 2025-06-20 20:28:00   Ariana Grande      One Last Time     197266   
3 2025-06-20 20:32:00  Camila Cabello  Never Be the Same     226974   
4 2025-06-20 20:36:00    James Arthur      Car's Outside     248373   

   minutes_played        date  year  month  hour weekday  \
0        3.479767  2025-06-20  2025      6    20  Friday   
1        4.316450  2025-06-20  2025      6    20  Friday   
2        3.287767  2025-06-20  2025      6    20  Friday   
3        3.782900  2025-06-20  2025      6    20  Friday   
4        4.139550  2025-06-20  2025      6    20  Friday   

                            track_key  
0            charlie puth - attention  
1             ed sheeran - p

## 9. Streaming Summary
### Summarize Streaming History per Track
Collapse the streaming history from one row per individual play down to one row per unique track calculating total play count, total minutes listened and the first/last time each track was played.

In [26]:
# Streaming Summary
streaming_summary = (
    streaming_df
    .groupby("track_key")
    .agg(
        artist_name=("artist_name", "first"),
        track_name=("track_name", "first"),
        play_count=("track_key", "count"),
        total_minutes_played=("minutes_played", "sum"),
        first_played=("end_time", "min"),
        last_played=("end_time", "max")
    )
    .reset_index()
)

streaming_summary.head()

,track_key,artist_name,track_name,play_count,total_minutes_played,first_played,last_played
0,(((()))) - novacane - 8d audio,(((()))),Novacane - 8D Audio,2,1.063917,2026-01-12 05:25:00,2026-02-11 08:05:00
1,03’babyshay - dreamz & nightmarez - wthelly?,03’BabyShay,Dreamz & NightMarez - Wthelly?,1,0.032167,2026-04-23 20:13:00,2026-04-23 20:13:00
2,5 seconds of summer - youngblood,5 Seconds of Summer,Youngblood,9,30.512617,2025-07-21 07:39:00,2026-02-13 16:09:00
3,50 cent - baby by me,50 Cent,Baby By Me,7,16.763717,2026-03-13 05:50:00,2026-06-16 11:15:00
4,50 cent - just a lil bit,50 Cent,Just A Lil Bit,7,8.813900,2026-03-13 05:56:00,2026-06-16 12:16:00


## 10. Playlist Summary
### Summarize Playlist Membership per Track
Collapse the playlist table down to one row per track, counting how many distinct playlists each track appears in and listing the names of those playlists.

In [27]:
# Playlist Summary
playlist_summary = (
    playlist_df
    .groupby("track_key")
    .agg(
        playlist_count=("playlist_name", "nunique"),
        playlists=("playlist_name", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

playlist_summary.head()

,track_key,playlist_count,playlists
0,5 seconds of summer - youngblood,1,Life
1,50 cent - baby by me,1,R&B
2,50 cent - just a lil bit,1,"Volume up ,logic down"
3,5star - cmonnn (hit it one time) (pt. 2) (feat...,3,"Dance away, Life, Volume up ,logic down"
4,aaryan shah - renegade,2,"1 A.M, ❤️😍"


## 11. Library Summary
### Summarize Library Status per Track
Reduce the library table to just the two columns needed downstream —`track_key` and `in_library` — removing any duplicate rows.

In [28]:
# Library Summary
library_summary = (
    library_df[["track_key", "in_library"]]
    .drop_duplicates()
)

library_summary.head()

,track_key,in_library
0,zoe clark - not to be dramatic,True
1,rumi - free,True
2,yuji - old love,True
3,khim - 10s,True
4,wale - bad (feat. rihanna) - remix,True


## 12. Merge Tables
### Combine Everything into One Master Table
Outer-merge the streaming summary, playlist summary and library summary on `track_key` so that every track the user has ever streamed, playlisted or saved ends up as a single row in one master `tracks_df` table.

In [29]:
# Merge Tables
tracks_df = streaming_summary.merge(
    playlist_summary,
    on="track_key",
    how="outer"
)

tracks_df = tracks_df.merge(
    library_summary,
    on="track_key",
    how="outer"
)

tracks_df.head()


,track_key,artist_name,track_name,play_count,total_minutes_played,first_played,last_played,playlist_count,playlists,in_library
0,(((()))) - novacane - 8d audio,(((()))),Novacane - 8D Audio,2.0,1.063917,2026-01-12 05:25:00,2026-02-11 08:05:00,NaN,NaN,True
1,03’babyshay - dreamz & nightmarez - wthelly?,03’BabyShay,Dreamz & NightMarez - Wthelly?,1.0,0.032167,2026-04-23 20:13:00,2026-04-23 20:13:00,NaN,NaN,NaN
2,5 seconds of summer - youngblood,5 Seconds of Summer,Youngblood,9.0,30.512617,2025-07-21 07:39:00,2026-02-13 16:09:00,1.0,Life,True
3,50 cent - baby by me,50 Cent,Baby By Me,7.0,16.763717,2026-03-13 05:50:00,2026-06-16 11:15:00,1.0,R&B,True
4,50 cent - just a lil bit,50 Cent,Just A Lil Bit,7.0,8.813900,2026-03-13 05:56:00,2026-06-16 12:16:00,1.0,"Volume up ,logic down",NaN


## 13. Fill Missing Values in Merged Tracks DataFrame
### Fill Gaps Created by the Merge
Because the merge was an outer join, some tracks will be missing values for fields they didn't have (e.g.a playlisted-but-never-streamed track has no play count). Fill those gaps with sensible defaults — 0 plays,0 minutes, "Not in any playlist", and `in_library = False` — so the table is complete and analysis-ready.

In [31]:
# Fill Missing Values in Merged Tracks DataFrame
tracks_df["play_count"] = tracks_df["play_count"].fillna(0).astype(int)
tracks_df["total_minutes_played"] = tracks_df["total_minutes_played"].fillna(0)
tracks_df["playlist_count"] = tracks_df["playlist_count"].fillna(0).astype(int)
tracks_df["playlists"] = tracks_df["playlists"].fillna("Not in any playlist")
tracks_df["in_library"] = tracks_df["in_library"].fillna(False)

print("Tracks Shape",tracks_df.shape)
print("Tracks Preview:\n",tracks_df.head())


Tracks Shape (2205, 10)
Tracks Preview:
                                       track_key          artist_name  \
0                (((()))) - novacane - 8d audio             (((())))   
1  03’babyshay - dreamz & nightmarez - wthelly?          03’BabyShay   
2              5 seconds of summer - youngblood  5 Seconds of Summer   
3                          50 cent - baby by me              50 Cent   
4                      50 cent - just a lil bit              50 Cent   

                       track_name  play_count  total_minutes_played  \
0             Novacane - 8D Audio           2              1.063917   
1  Dreamz & NightMarez - Wthelly?           1              0.032167   
2                      Youngblood           9             30.512617   
3                      Baby By Me           7             16.763717   
4                  Just A Lil Bit           7              8.813900   

         first_played         last_played  playlist_count  \
0 2026-01-12 05:25:00 2026-02-11 08:05

## 14. Save Cleaned DataFrames to CSV
### Export Cleaned Data
Save the cleaned streaming, playlist and library tables, along with the final merged `tracks_df` summary,as CSV files so they can be reused directly in the EDA and recommendation-engine notebooks without repeating the cleaning steps.

In [ ]:
# Save Cleaned DataFrames to CSV
streaming_df.to_csv(r"Data/Cleaned/Streaming_history_cleaned.csv", index=False)
playlist_df.to_csv(r"Data/Cleaned/Playlist_cleaned.csv", index=False)
library_df.to_csv(r"Data/Cleaned/Library_cleaned.csv", index=False)
tracks_df.to_csv(r"Data/Cleaned/Tracks_summary.csv", index=False)
